In [1]:
#!pip install optuna

In [2]:
import optuna 
import torch
from train_generalized_earlystopping import train, bce_loss, dice_loss, bce_dice_loss, focal_loss, tversky_loss
from data import load_mri_dataframe, get_dataloaders
from BaselineUNetParams import BaselineUNet

In [3]:
def objective(trial):

    # 1. Define hyperparameters to be optimized
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [4, 8, 16])
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD"])
    num_layers = trial.suggest_int("num_layers", 2, 8) # Number of encoder/decoder layers
    num_filters = trial.suggest_categorical("num_filters", [8, 16, 32])
    lr_step_size = trial.suggest_int("step_size", 3, 7)  # for lr scheduler
    lr_gamma = trial.suggest_float("gamma", 0.1, 0.5)  # for lr scheduler

    # 2. Load data
    df = load_mri_dataframe()
    train_loader, val_loader = get_dataloaders(df, batch_size=batch_size, omit_empty_masks=True)

    # 3. Init model, optimizer, loss function
    model = BaselineUNet(num_layers=num_layers, num_filters=num_filters)
    device = torch.device("cuda")
    model.to(device)

    if optimizer_name == "Adam":
        optimizer_class = torch.optim.Adam
    else:
        optimizer_class = torch.optim.SGD

    #loss_fn_dict = {
    #    "bce_loss": bce_loss,
    #    "dice_loss": dice_loss,
    #    "bce_dice_loss": bce_dice_loss,
    #    "focal_loss": focal_loss,
    #    "tversky_loss": tversky_loss,
    #}
    #loss_fn = loss_fn_dict[loss_fn_name]

    # 4. Train model
    trained_model, results = train(
        model,
        train_loader,
        val_loader,
        device,
        lr=lr,
        optimizer_class=optimizer_class,
        loss_fn=bce_loss,
        epochs=300,
        lr_sched_cls=torch.optim.lr_scheduler.StepLR,
        lr_sched_kwargs={"step_size": lr_step_size, "gamma": lr_gamma},
    )

    # 5. Evaluate
    val_dice = results["history"]["val_dice"][-1]  # Last epoch dice score
    return val_dice

In [4]:
study = optuna.create_study(direction="maximize")  # Trial goal: maximize Dice score //TODO: combine with PatientPruner
study.optimize(objective, n_trials=40)  # No. of trials to run                       //reason: optune will penalize trial if stopped early

trial = study.best_trial

print("\nBest Validation Dice Score: {}".format(trial.value))

print("\nWith Parameters:")
for key, value in trial.params.items():
    print("   {}: {}".format(key, value))

[I 2025-06-19 17:15:26,780] A new study created in memory with name: no-name-a6d089d9-dce4-4154-90d8-712fd7e2769b


[Data] Train images: 1093 ; Val images: 280


[I 2025-06-19 17:16:19,376] Trial 0 finished with value: 1.6386562234111629e-09 and parameters: {'lr': 0.00541930189707534, 'batch_size': 8, 'optimizer': 'SGD', 'num_layers': 5, 'num_filters': 8, 'step_size': 6, 'gamma': 0.27017097008750857}. Best is trial 0 with value: 1.6386562234111629e-09.


Stopped early at epoch 15!
[Data] Train images: 1093 ; Val images: 280


[W 2025-06-19 17:21:04,385] Trial 1 failed with parameters: {'lr': 0.0005208988125691879, 'batch_size': 16, 'optimizer': 'Adam', 'num_layers': 7, 'num_filters': 32, 'step_size': 7, 'gamma': 0.17112788007676674} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\Name\AppData\Local\Programs\Python\Python313\Lib\site-packages\optuna\study\_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\Name\AppData\Local\Temp\ipykernel_7264\3936168111.py", line 36, in objective
    trained_model, results = train(
                             ~~~~~^
        model,
        ^^^^^^
    ...<8 lines>...
        lr_sched_kwargs={"step_size": lr_step_size, "gamma": lr_gamma},
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "c:\Users\Name\IdeaProjects\AILab\Brain-Tumor-Semantic-Segmentation\train_generalized_earlystopping.py", line 173, in train
    running_loss += loss.item()


KeyboardInterrupt: 